In [13]:
# Use California Housing dataset as implemented dataset
from sklearn.datasets import fetch_california_housing
import torch
import sys
sys.path.append('/teamspace/studios/this_studio/d2l-notes')

from sarahdl_baseclass import Model, DataLoader, Trainer

data = fetch_california_housing()
x = torch.tensor(data.data, dtype=torch.float32)    
y = torch.tensor(data.target, dtype=torch.float32)  

In [14]:
# Implement get_dataloader
from sklearn.datasets import fetch_california_housing
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader
from sklearn.preprocessing import StandardScaler


class CAHouseData(DataLoader):
    def __init__(self,batch_size):
        super().__init__(batch_size)
        data = fetch_california_housing()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data.data)
        x = torch.tensor(X_scaled, dtype=torch.float32)
        y = torch.tensor(data.target, dtype = torch.float32)
        self.dataset = TensorDataset(x,y)

    def get_dataloader(self):
        return TorchDataLoader(self.dataset,batch_size= self.batch_size, shuffle= True)

In [15]:
# Classes for linear regression analysis
# Inherits the classes from sarahdl_baseclass

class LinearRegression(Model):
    def __init__(self, input_num, lr, sigma = 0.01):
        super().__init__(output_num = 1)
        self.w = torch.normal(0,sigma,(input_num,1),requires_grad = True)
        self.b = torch.zeros(1,requires_grad = True)
        self.lr = lr

    def forward(self,x):
        return torch.matmul(x,self.w) + self.b

    def loss(self, y_hat, y):
        l = (y_hat - y.reshape(y_hat.shape)) **2 / 2
        return l.mean()

    def configure_optimizer(self):
        return SGD([self.w, self.b], self.lr)

In [16]:
# SGD Optimizer
class SGD:
    def __init__(self,params,lr):
        self.params = params
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for params in self.params:
                params -= self.lr * params.grad

    def zero_grad(self):
        for params in self.params:
            if params.grad is not None:
                params.grad.zero_()


In [17]:
# Trainer for linear regression
class LRTrainer(Trainer):
    def __init__(self,max_epoch):
        super().__init__(max_epoch)

    def fit_epoch(self):
        for batch in self.dataloader:
            loss = self.model.training_step(batch)
            self.optim.zero_grad()
            loss.backward()
            self.optim.step()

        

In [18]:
# Create instances
srl_data = CAHouseData(batch_size= 32)
srl_model = LinearRegression(input_num= 8, lr = 0.03)
srl_trainer = LRTrainer(max_epoch=3)

srl_trainer.fit(srl_model,srl_data)

In [19]:
print(f'w: {srl_model.w}')
print(f'b: {srl_model.b}')

w: tensor([[ -2.4112],
        [ 33.1004],
        [ 38.8113],
        [  3.2223],
        [ 88.4401],
        [602.0040],
        [ -4.6260],
        [-13.6603]], requires_grad=True)
b: tensor([16.5261], requires_grad=True)


In [20]:
srl_model = LinearRegression(input_num=8, lr=0.001)

In [21]:
srl_trainer.fit(srl_model,srl_data)

In [22]:
print(f'w: {srl_model.w}')
print(f'b: {srl_model.b}')

w: tensor([[ 0.6668],
        [ 0.1580],
        [ 0.0511],
        [-0.0415],
        [ 0.0023],
        [-0.0304],
        [-0.1739],
        [-0.1259]], requires_grad=True)
b: tensor([1.7698], requires_grad=True)


In [23]:
with torch.no_grad():
    total_loss = 0
    for batch in srl_data.get_dataloader():
        x, y = batch[0], batch[1]
        y_hat = srl_model(x)
        loss = srl_model.loss(y_hat, y)
        total_loss += loss.item()
    print(f'Average loss: {total_loss / srl_trainer.num_batches:.4f}')

Average loss: 0.3638


In [24]:
import math
rmse = math.sqrt(2 * 0.3638)  
print(f'RMSE: {rmse:.4f}')

RMSE: 0.8530
